<a href="https://colab.research.google.com/github/Abeer-Ahma317/Arabic-Chat-Bot-/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**instal library**




In [1]:
!pip install transformers pandas langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
!pip install transformers accelerate bitsandbytes
!pip install langchain langchain-community
!pip install pandas torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 18.0 MB/s eta 0:00:00


**DB**

In [3]:
# جلب الأسماء من قاعدة البيانات
import sqlite3
import pandas as pd
conn = sqlite3.connect("/content/Student.db")
cursor = conn.cursor()

first_names = [row[0] for row in cursor.execute("SELECT DISTINCT first_name FROM Students").fetchall()]
father_names = [row[0] for row in cursor.execute("SELECT DISTINCT father_name FROM Students").fetchall()]
grandfather_names = [row[0] for row in cursor.execute("SELECT DISTINCT grandfather_name FROM Students").fetchall()]
family_names = [row[0] for row in cursor.execute("SELECT DISTINCT family_name FROM Students").fetchall()]

db_entities = {
    "FIRST_NAME": {name: name for name in first_names},
    "FATHER_NAME": {name: name for name in father_names},
    "GRANDFATHER_NAME": {name: name for name in grandfather_names},
    "FAMILY_NAME": {name: name for name in family_names}
}

In [4]:
import sqlite3
import pandas as pd
import re
from typing import Dict, List, Tuple, Any, Optional
import numpy as np

# تجربة استيراد المكتبات الإضافية
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("Transformers not available!")
    print("Install with: pip install transformers torch accelerate bitsandbytes")
    TRANSFORMERS_AVAILABLE = False


# Global variables - التركيز على SQLCoder
SQLCODER_MODEL = None
SQLCODER_TOKENIZER = None
SQLCODER_PIPELINE = None
CONN = None
SCHEMA = ""



**Model(SQLCoder)**

In [5]:
def load_sqlcoder_model():
    """تحميل SQLCoder-7B - نموذج متخصص في SQL"""
    global SQLCODER_MODEL, SQLCODER_TOKENIZER, SQLCODER_PIPELINE

    print("Loading SQLCoder-7B Model...")

    if not TRANSFORMERS_AVAILABLE:
        print("Cannot load SQLCoder: Transformers not available")
        return False

    try:
        import torch
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {device}")

        model_name = "defog/sqlcoder-7b-2"
        print(f"Loading SQLCoder from: {model_name}")
        print("This will download ~13GB on first run...")

        # تحميل Tokenizer
        SQLCODER_TOKENIZER = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )

        # إضافة pad_token إذا لم يكن موجود
        if SQLCODER_TOKENIZER.pad_token is None:
            SQLCODER_TOKENIZER.pad_token = SQLCODER_TOKENIZER.eos_token
            SQLCODER_TOKENIZER.pad_token_id = SQLCODER_TOKENIZER.eos_token_id

        # تحميل Model مع 8-bit quantization لتوفير الذاكرة
        SQLCODER_MODEL = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
            load_in_8bit=True  # استخدام 8-bit لتقليل الذاكرة
        )

        print("SQLCoder Model loaded successfully")

        # إنشاء Pipeline
        print("Creating SQLCoder pipeline...")
        SQLCODER_PIPELINE = pipeline(
            "text-generation",
            model=SQLCODER_MODEL,
            tokenizer=SQLCODER_TOKENIZER,
            max_new_tokens=300,
            temperature=0.0,  # SQLCoder يعمل أفضل بدون عشوائية
            do_sample=False,
            return_full_text=False,
            pad_token_id=SQLCODER_TOKENIZER.pad_token_id,
            eos_token_id=SQLCODER_TOKENIZER.eos_token_id
        )

        print("SQLCoder Pipeline created successfully")
        return True

    except Exception as e:
        print(f"Error loading SQLCoder: {e}")
        print("Make sure you have: transformers, accelerate, bitsandbytes installed")
        return False

def test_sqlcoder():
    """اختبار SQLCoder"""
    if not SQLCODER_PIPELINE:
        return False

    try:
        test_prompt = """### Task
Generate a SQL query to answer: How many students are there?

### Database Schema
CREATE TABLE Students (id INTEGER, name TEXT);

### Answer
```sql
"""

        response = SQLCODER_PIPELINE(test_prompt, max_new_tokens=50)

        if isinstance(response, list) and len(response) > 0:
            result = response[0].get('generated_text', '')
            print(f"SQLCoder Test Response: {result[:80]}...")
            return True
        else:
            print(f"SQLCoder Test Response: {response}")
            return True

    except Exception as e:
        print(f"SQLCoder test failed: {e}")
        return False

**7 Layer Input Enhancement**

In [16]:
def layer1_text_normalization(question: str) -> Dict[str, Any]:
    """الطبقة 1: تطبيع النص العربي"""

    def normalize_arabic(text: str) -> str:
        if pd.isna(text):
            return ""
        text = str(text).strip()
        text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
        text = text.replace("ؤ", "و").replace("ئ", "ي").replace("ة", "ه")
        text = text.replace("ـ", "")
        return text

    normalized = normalize_arabic(question)
    tokens = [word.strip() for word in normalized.split() if word.strip()]

    return {
        'original': question,
        'normalized': normalized,
        'tokens': tokens,
        'clean_text': ' '.join(tokens)
    }

def layer2_intent_classification(processed_text: Dict[str, Any]) -> Dict[str, Any]:
    """الطبقة 2: تصنيف النية"""

    text = processed_text['normalized'].lower()

    intent_patterns = {
        'COUNT_QUERY': ['كم', 'عدد', 'كام'],
        'LIST_QUERY': ['اسماء', 'ما هي', 'اعرض', 'اظهر'],
        'AGGREGATE_QUERY': ['متوسط', 'اعلى', 'اقل', 'مجموع'],
        'FILTER_QUERY': ['من', 'في', 'التي', 'الذين'],
        'JOIN_QUERY': ['معهد', 'موقع', 'منطقة', 'اقليم']
    }

    detected_intent = 'GENERAL_QUERY'
    confidence = 0.0

    for intent, keywords in intent_patterns.items():
        matches = sum(1 for keyword in keywords if keyword in text)
        if matches > 0:
            current_confidence = matches / len(keywords)
            if current_confidence > confidence:
                detected_intent = intent
                confidence = current_confidence

    return {
        'intent': detected_intent,
        'confidence': confidence
    }

def layer3_entity_recognition(processed_text: Dict,db_entities: dict = None) -> dict:
    """الطبقة 3: استخراج الكيانات"""
    if isinstance(processed_text, str):
        processed_text = {"normalized": processed_text}

    text = processed_text['normalized']
    entities = []

   # قاموس الكيانات
    entity_patterns = {
        "national_id": r"\b\d{10}\b",         # الرقم الوطني - 10 أرقام
        "phone_number": r"\b96\d{10}\b",       # رقم الهاتف الأردني
        "birth_date": r"\b\d{4}-\d{2}-\d{2}\b",  # تاريخ الميلاد (yyyy-mm-dd)
        "exam_grade": r"\b\d{1,3}\b",         # علامة امتحان (0-100 أو أكثر)
        'REGION': {
            'الشمالي': 'الإقليم الشمالي',
            'شمالي': 'الإقليم الشمالي',
            'الجنوبي': 'الإقليم الجنوبي',
            'جنوبي': 'الإقليم الجنوبي',
            'الأوسط': 'الإقليم الأوسط',
            'اوسط': 'الإقليم الأوسط',
            'أوسط': 'الإقليم الأوسط',
            'الشمال': 'الإقليم الشمالي',
            'الجنوب': 'الإقليم الجنوبي',
            'الوسط': 'الإقليم الأوسط'
        },
        'PROFESSION': {
            'ميكانيكي': 'ميكانيكي',
            'مشغل': 'مشغل',
            'حلاق': 'حلاق',
            'تمديدات': 'تمديدات',
            'خياط': 'خياط',
            'دهان': 'دهان' ,
            'مجهز': 'مجهز',
            'مركب': 'مركب',
            'لحيم': 'لحيم',
            'حداد': 'حداد',
            'نجار': 'نجار',
            'كهربائي': 'كهربائي',
            'داعم': 'داعم',
            'قصير': 'قصير' ,
            'بليط': 'بليط',
            'مزارع': 'مزارع',
            'التسويق': 'التسويق' ,
            'طاقة شمسية': 'طاقة شمسية'
        },
        'GENDER': {
            'ذكر': 'ذكر',
            'ذكور': 'ذكر',
            'انثى': 'أنثى',
            'اناث': 'أنثى',
            'أنثى': 'أنثى'
        },
        'QUALIFICATION': {
            'بكالوريوس': 'بكالوريوس',
            'دبلوم': 'دبلوم',
            'ماجستير': 'ماجستير',
            'دكتوراه': 'دكتوراه'
        },
        'AREA': {
            'الزرقاء': 'الزرقاء',
            'الموقر': 'الموقر',
            'مادبا': 'مادبا',
            'ماركا الصناعي': 'ماركا الصناعي',
            'ذيبان ': 'ذيبان',
            'الكرك': 'الكرك',
            'معان': 'معان' ,
            'الطفيلة': 'الطفيلة',
            'القويرة': 'القويرة',
            'الريشة': 'الريشة ',
            'السرحان': 'السرحان',
            'عجلون': 'عجلون',
            'الرمثا': 'الرمثا',
            'جرش': 'جرش',
            'الكورة': 'الكورة'
        },
        'INSTITUTES': {
            'معهد الزرقاء': 'معهد الزرقاء',
            'معهد الموقر': 'معهد الموقر',
            'معهد مادبا': 'معهد مادبا',
            'معهد ماركا الصناعي': 'معهد ماركا الصناعي',
            'معهد ذيبان ': 'معهد ذيبان',
            'معهد الكرك': 'معهد الكرك',
            'معهد معان': 'معهد معان' ,
            'معهد الطفيلة': 'معهد الطفيلة',
            'معهد القويرة': 'القويرة',
            'معهد الريشة': 'الريشة ',
            'معهد السرحان': 'السرحان',
            'معهد عجلون': 'عجلون',
            'معهد الرمثا': 'الرمثا',
            'معهد جرش': 'جرش',
            'معهد الكورة': 'الكورة'
        }
    }
    if db_entities:
        entity_patterns.update(db_entities)

    # البحث عن الكيانات
    for entity_type, patterns in entity_patterns.items():
        if isinstance(patterns, str):  # regex
            match = re.search(patterns, text)
            if match:
                entities.append({'type': entity_type, 'value': match.group(), 'original': match.group()})
        elif isinstance(patterns, dict):  # dictionary
            for pattern, standard_value in patterns.items():
                if pattern in text:
                    entities.append({'type': entity_type, 'value': standard_value, 'original': pattern})

    return {
        'entities': entities,
        'entity_count': len(entities),
        'entity_types': list(set([e['type'] for e in entities]))
    }

def layer4_context_analysis(processed_text: Dict[str, Any], intent: Dict[str, Any], entities: Dict[str, Any]) -> Dict[str, Any]:
    """الطبقة 4: تحليل السياق"""

    text = processed_text['normalized']

    table_indicators = {
        'Students': ['طالب', 'طلاب', 'دارس'],
        'Locations': ['منطقة', 'اقليم', 'موقع'],
        'Institutes': ['معهد', 'معاهد', 'مؤسسة'],
        'Exams': ['علامة','علامته', 'درجة', 'امتحان'] }

    required_tables = []
    for table, indicators in table_indicators.items():
        if any(indicator in text for indicator in indicators):
            required_tables.append(table)

    if not required_tables:
        required_tables = ['Students']

    join_needed = len(required_tables) > 1

    return {
        'required_tables': required_tables,
        'join_needed': join_needed,
        'complexity_level': 'HIGH' if join_needed else 'LOW'
    }

def layer5_schema_mapping(context: Dict[str, Any]) -> Dict[str, Any]:
    """الطبقة 5: ربط بالسكيما"""
    global SCHEMA

    schema_info = {}
    current_table = None

    for line in SCHEMA.split('\n'):
        line = line.strip()
        if line.startswith('CREATE TABLE'):
            current_table = line.split()[2].strip('(')
            schema_info[current_table] = []
        elif current_table and line and not line.startswith('CREATE'):
            schema_info[current_table].append(line)

    relevant_schema = {}
    for table in context['required_tables']:
        if table in schema_info:
            relevant_schema[table] = schema_info[table]

    return {
        'relevant_schema': relevant_schema,
        'full_schema': schema_info
    }

def layer6_prompt_optimization(question: str, intent: Dict, entities: Dict, context: Dict, schema_mapping: Dict) -> str:
    """الطبقة 6: تحسين البرومبت لـ SQLCoder"""

    # SQLCoder format
    prompt = f"""### Task
Generate a SQL query to answer the following Arabic question: `{question}`

### Database Schema
{SCHEMA}

### Additional Context
- Question Type: {intent['intent']}
- Entities: {', '.join([e['value'] for e in entities['entities']]) if entities['entities'] else 'None'}
- Tables Needed: {', '.join(context['required_tables'])}
- For Arabic text matching, use LIKE '%text%'
- IMPORTANT: Do NOT use ILIKE. Use only LIKE. SQLite does not support ILIKE.
  Example: use "WHERE column LIKE '%text%'" instead of "ILIKE".

- Relationships:
- Relationships:
  * Students.residence_location_id → Locations.id
  * Students.institute_id → Institutes.id
  * Exams.student_id → Students.national_id

### Answer
Given the database schema, here is the SQL query that answers `{question}`:
```sql
"""

    return prompt

def layer7_response_processing(raw_response: str) -> str:
    """الطبقة 7: معالجة استجابة SQLCoder"""

    if not raw_response:
        return None

    # استخراج من code block
    sql_block_match = re.search(r'```sql\n(.*?)```', raw_response, re.DOTALL | re.IGNORECASE)
    if sql_block_match:
        sql = sql_block_match.group(1).strip()
        return sql.replace(';', '').strip()

    # استخراج SELECT statement
    lines = raw_response.split('\n')
    for line in lines:
        line = line.strip()
        if line.upper().startswith('SELECT'):
            return line.replace(';', '').strip()

    # Regex fallback
    sql_match = re.search(r'SELECT[^;]*', raw_response, re.IGNORECASE | re.DOTALL)
    if sql_match:
        return sql_match.group(0).strip().replace(';', '')

    return None


**MAIN PROCESSING PIPELINE**

In [13]:
def process_arabic_question_with_sqlcoder(question: str) -> Dict[str, Any]:
    """معالجة السؤال عبر 7 طبقات ثم SQLCoder"""

    print(f"Processing through 7-layer enhancement for SQLCoder")
    print(f"Question: {question}")
    print("=" * 60)

    # Layer 1
    print("Layer 1: Text Normalization")
    layer1_result = layer1_text_normalization(question)
    print(f"  Normalized: '{layer1_result['normalized']}'")

    # Layer 2
    print("Layer 2: Intent Classification")
    layer2_result = layer2_intent_classification(layer1_result)
    print(f"  Intent: {layer2_result['intent']}")

    # Layer 3
    print("Layer 3: Entity Recognition")
    layer3_result = layer3_entity_recognition(layer1_result)
    print(f"  Entities: {layer3_result['entity_count']} found")

    # Layer 4
    print("Layer 4: Context Analysis")
    layer4_result = layer4_context_analysis(layer1_result, layer2_result, layer3_result)
    print(f"  Tables: {layer4_result['required_tables']}")

    # Layer 5
    print("Layer 5: Schema Mapping")
    layer5_result = layer5_schema_mapping(layer4_result)
    print(f"  Schema mapped: {len(layer5_result['relevant_schema'])} tables")

    # Layer 6
    print("Layer 6: Prompt Optimization")
    optimized_prompt = layer6_prompt_optimization(question, layer2_result, layer3_result, layer4_result, layer5_result)
    print(f"  Prompt created ({len(optimized_prompt)} chars)")

    # Send to SQLCoder
    print("Sending to SQLCoder Model...")
    raw_response = send_to_sqlcoder(optimized_prompt)
    print(f"  SQLCoder response: {str(raw_response)[:100]}...")

    # Layer 7
    print("Layer 7: Response Processing")
    final_sql = layer7_response_processing(raw_response)
    print(f"  Final SQL: {final_sql}")

    return {
        'question': question,
        'final_sql': final_sql,
        'success': final_sql is not None,
        'raw_response': raw_response
    }

def send_to_sqlcoder(prompt: str) -> str:
    """إرسال لـ SQLCoder"""
    global SQLCODER_PIPELINE

    if not SQLCODER_PIPELINE:
        print("SQLCoder not available!")
        return None

    try:
        response = SQLCODER_PIPELINE(prompt, max_new_tokens=300)
        if isinstance(response, list) and len(response) > 0:
            return response[0].get('generated_text', '')
        return str(response)

    except Exception as e:
        print(f"Error calling SQLCoder: {e}")
        return None


In [20]:
def execute_enhanced_query(question: str) -> pd.DataFrame:
    """تنفيذ الاستعلام المحسن"""
    global CONN

    result = process_arabic_question_with_sqlcoder(question)

    if not result['success'] or not result['final_sql']:
        print("Failed to generate SQL")
        return pd.DataFrame()

    try:
        # ✅ تصحيح تلقائي قبل التنفيذ
        sql_query = result['final_sql']
        if "ILIKE" in sql_query:
            import re
            sql_query = sql_query.replace("ILIKE", "LIKE")
            print("⚙️ Auto-fixed: Replaced ILIKE → LIKE")

        # تنفيذ الاستعلام بعد التصحيح
        df = pd.read_sql(sql_query, CONN)
        print(f"Results: {len(df)} rows")
        print(df)
        return df

    except Exception as e:
        print(f"SQL execution error: {e}")
        print(f"SQL was: {result['final_sql']}")
        return pd.DataFrame()

def run_enhanced_tests():
    """تشغيل اختبارات محسنة"""

    test_questions = [
        "كم عدد الذكور و من الاقليم الاوسط  ؟",
        "كم طالب عمره اقل من 30 ؟",
        "من الطالب الي رقم هاتفه 962917404542",
        "ما أسماء المعاهد في الاقليم االجنوبي ؟",
        "كم طالب مهنته مركب "
    ]

    print("Running Tests with SQLCoder")
    print("=" * 50)

    for i, question in enumerate(test_questions, 1):
        print(f"\nTest {i}/{len(test_questions)}")
        try:
            result = execute_enhanced_query(question)
            if not result.empty:
                print("Test passed")
        except Exception as e:
            print(f"Test failed: {e}")
        print("=" * 50)

# =============================================================================
# SYSTEM INITIALIZATION
# =============================================================================

def initialize_sqlcoder_system(db_path: str) -> bool:
    """تهيئة النظام مع SQLCoder"""
    global CONN, SCHEMA

    print("Initializing SQLCoder-7B System with 7-Layer Enhancement")
    print("=" * 60)

    # Load SQLCoder
    if not load_sqlcoder_model():
        print("Cannot proceed without SQLCoder model!")
        return False

    # Test SQLCoder
    if not test_sqlcoder():
        print("SQLCoder model not working properly!")
        return False

    # Load Database
    try:
        import os
        if not os.path.exists(db_path):
            print(f"Database not found: {db_path}")
            return False

        CONN = sqlite3.connect(db_path)
        print("Database connected")

        # Build Schema in SQLCoder format
        tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", CONN)
        SCHEMA = ""

        for table in tables['name']:
            if table != 'sqlite_sequence':
                cols = pd.read_sql(f"PRAGMA table_info([{table}]);", CONN)
                SCHEMA += f"\nCREATE TABLE {table} (\n"
                col_defs = []
                for _, col in cols.iterrows():
                    col_defs.append(f"    {col['name']} {col['type']}")
                SCHEMA += ",\n".join(col_defs)
                SCHEMA += "\n);\n"

        print("Schema built (SQLCoder format)")

    except Exception as e:
        print(f"Database error: {e}")
        return False

    print("SQLCoder system ready!")
    return True

# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("SQLCoder-7B System with 7-Layer Enhancement")
    print("=" * 60)

    # Database path
    DB_PATH = "Student.db"  # Update this

    # Initialize
    if not initialize_sqlcoder_system(DB_PATH):
        print("System initialization failed!")
        exit(1)

    print("\n" + "="*60)
    print("Running Tests")
    run_enhanced_tests()

    print("\n" + "="*60)
    print("System ready!")
    print("Use execute_enhanced_query('your question') for queries")

SQLCoder-7B System with 7-Layer Enhancement
Initializing SQLCoder-7B System with 7-Layer Enhancement
Loading SQLCoder-7B Model...
Using device: cuda
Loading SQLCoder from: defog/sqlcoder-7b-2
This will download ~13GB on first run...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


SQLCoder Model loaded successfully
Creating SQLCoder pipeline...
SQLCoder Pipeline created successfully
SQLCoder Test Response:  SELECT COUNT(s.id) FROM Students s;...
Database connected
Schema built (SQLCoder format)
SQLCoder system ready!

Running Tests
Running Tests with SQLCoder

Test 1/5
Processing through 7-layer enhancement for SQLCoder
Question: كم عدد الذكور و من الاقليم الاوسط  ؟
Layer 1: Text Normalization
  Normalized: 'كم عدد الذكور و من الاقليم الاوسط  ؟'
Layer 2: Intent Classification
  Intent: COUNT_QUERY
Layer 3: Entity Recognition
  Entities: 2 found
Layer 4: Context Analysis
  Tables: ['Locations']
Layer 5: Schema Mapping
  Schema mapped: 1 tables
Layer 6: Prompt Optimization
  Prompt created (1351 chars)
Sending to SQLCoder Model...
  SQLCoder response:  SELECT COUNT(DISTINCT s.national_id) AS male_students_from_middle_region FROM Students s JOIN Locat...
Layer 7: Response Processing
  Final SQL: SELECT COUNT(DISTINCT s.national_id) AS male_students_from_middle_regi